In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
from moviepy.video.io.VideoFileClip import VideoFileClip
import os
import glob
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import math 
import re 

import imageio_ffmpeg
import os

os.environ["IMAGEIO_FFMPEG_EXE"] = imageio_ffmpeg.get_ffmpeg_exe()


d:\Users\strat\anaconda3\envs\lighthouse_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'sentence_transformers'

In [ ]:

model_path = "HuggingFaceTB/SmolVLM2-256M-Video-Instruct"

processor = AutoProcessor.from_pretrained(model_path)

model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.float32,
    _attn_implementation="eager"
).to("cuda")

model.eval()



SmolVLMForConditionalGeneration(
  (model): SmolVLMModel(
    (vision_model): SmolVLMVisionTransformer(
      (embeddings): SmolVLMVisionEmbeddings(
        (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
        (position_embedding): Embedding(1024, 768)
      )
      (encoder): SmolVLMEncoder(
        (layers): ModuleList(
          (0-11): 12 x SmolVLMEncoderLayer(
            (self_attn): SmolVLMVisionAttention(
              (k_proj): Linear(in_features=768, out_features=768, bias=True)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): SmolVLMVisionMLP(
              (activation_fn): PytorchGELUTanh()
              (fc1): Linear(in_features=768, out_

In [ ]:
# THIS HAS TO CHANGE - HEAVY EXPERIMENTATION
PROMPT_EVENT_ONLY = """ 
Describe the most unique actions in this video segment in one concise sentence. Focus on what is happening and who might be involved. 
"""

#Jesse:
#1. "Describe the single most important event in this video segment in one concise sentence. Focus on what is happening and who is involved.
#gave very repetitive answers, and often missed important events. Mainly has "man in white shirt speaking to camera...". 
#Could be seen as most "important" as it is visible the largest part of the video segments
#2. To get rid of this repitive stuff I tried to incorporate the events that are already outputted by the VLM in the prompt
#to give it a bit of context from what already has happened.
# Addition to 1: Prevent describing events that are already outputted, which are: + all_outputs descriptions so far. 
#This did not help at all, only made it worse. Less focus on describing right events and even more repitive events.
#Have a feeling this extra constraining is not working at all and we have to be clear in 1/2 sentence and being careful with the words we choose
#3. Describe the most unique actions in this video segment in one concise sentence. Focus on what is happening and who might be involved. 
#Really finds more distinctive events, does not keep describing the man talking to the camera


#Nikitas:
#"Describe the most relevant events in the video, list each event sequentially, using a numbered format. Describe it in terms of the actions different persons take" 

#Alex
#1) Retrieve all salient events in the video. Shit for both video 5 and video 6!
#2) Describe all the key events in the video. Much better for video 5 and incredibly better for video 6. A lot of yapping though and overlap.
#3) "Describe all the key events in the video." "Return only events that are important for understanding the video." 
# A bit better, but still hasn't captured salient events.

In [37]:

# this function runs the model per input chunk 
# need to define number of frames used per chunk + number of output tokens
def run_vlm_on_video(video_path, prompt, num_frames=8, max_new_tokens=128):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "video", "path": video_path},
                {"type": "text", "text": prompt}
            ]
        }
    ]




    inputs = processor.apply_chat_template(
    messages,
    num_frames = num_frames,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    ).to(model.device) # Considering the size of the model, and the number of frames we are sampling are few. This is ultimately a choice that you must make
    
    
    
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens
        )

    generated_only = generated_ids[:, inputs["input_ids"].shape[1]:]

    answer = processor.batch_decode(
        generated_only,
        skip_special_tokens=True
    )[0]

    return answer.strip()


In [34]:
# splitting the entire video into chunks of lenght x seconds


def split_video_into_chunks(video_path, output_dir, chunk_length=8):
    os.makedirs(output_dir, exist_ok=True)

    # If chunks already exist, reuse them
    existing_chunks = sorted(glob.glob(os.path.join(output_dir, "chunk_*.mp4")))
    if existing_chunks:
        chunk_paths = []

        for chunk_path in existing_chunks:
            filename = os.path.basename(chunk_path)
            parts = filename.replace("chunk_", "").replace(".mp4", "").split("_")
            start = int(parts[0])
            end = int(parts[1])
            chunk_paths.append((chunk_path, start, end))

        print(f"Reusing {len(chunk_paths)} existing chunks from {output_dir}")
        return chunk_paths

    # Otherwise, create chunks
    video = VideoFileClip(video_path)
    # duration = int(video.duration)
    duration = video.duration

    chunk_paths = []

    for start in range(0, int(duration), chunk_length): # change to int(duration) from duration
        end = min(start + chunk_length, int(duration)) # change to int(duration)

        chunk_path = os.path.join(
            output_dir,
            f"chunk_{start:04d}_{end:04d}.mp4"
        )

        if hasattr(video, "subclipped"):
            clip = video.subclipped(start, end)
        else:
            clip = video.subclip(start, end)

        clip.write_videofile(
            chunk_path,
            codec="libx264",
            audio=False,
            logger=None
        )

        clip.close()
        chunk_paths.append((chunk_path, start, end))

    video.close()
    return chunk_paths

In [36]:

# CHOOSE VIDEO
video_number = 5
video_path = f"TVSum/video_{video_number}.mp4"


# PARAMETERS
# chunk length is the lenght in seconds of each chunk
# num frames is the number of frames per chunk
# max tokens is the upper bound on number of words per generated output for each chunk

chunk_length = 15
num_frames = 10
max_new_tokens = 128

output_dir = f"video_{video_number}_chunks_{chunk_length}s"

chunks = split_video_into_chunks(
    video_path,
    output_dir=output_dir,
    chunk_length= chunk_length# changed to 8 seconds
)

all_outputs = []


for chunk_path, start, end in chunks:

    output = run_vlm_on_video(
        chunk_path,
        PROMPT_EVENT_ONLY,
        num_frames=num_frames, 
        max_new_tokens=max_new_tokens # match it to length of 1-2 sentences
    )

    all_outputs.append({
        "chunk": chunk_path,
        "start": start,
        "end": end,
        "vlm_output": output
    })

    print(f"\nCHUNK {start}-{end} seconds")
    print(output)




Reusing 8 existing chunks from video_5_chunks_15s

CHUNK 0-15 seconds
A man in a white shirt stands in front of a GMC logo and says "Replacing Your Tires."

CHUNK 15-30 seconds
A man is working on a car tire, using a tool to remove the tire.

CHUNK 30-45 seconds
A hand holding a metal rod with a needle attached to it.

CHUNK 45-60 seconds
A person is holding a coin and explaining something about it.

CHUNK 60-75 seconds
A person is using a tool to remove a wheel from a truck.

CHUNK 75-90 seconds
A person is standing next to a car with a tire on it.

CHUNK 90-105 seconds
A man in a white shirt is shaking hands with another person in a black shirt.

CHUNK 105-111 seconds
A man in a white shirt stands in front of a GMC logo and says "GMC Certified Service" and "Replacing Your Tires."


In [38]:
def parse_vlm_events(text):
    # Expects the output per chunk (text is for chunk output) to be normal text descritpion
    # Should work also if each chunk gives multiple events

    parsed_events = []
    parsed_events.append({
            "event": text.strip(" ,.-"),   # remove comma, dot etc.
            "start_time": None,
            "end_time": None
    })

    return parsed_events

In [39]:
# go over all chunk outputs in chronological order 
# and store a list of salient event descriptions
predicted_events = []

for item in all_outputs:
    parsed = parse_vlm_events(item["vlm_output"])
    predicted_events.extend([e["event"] for e in parsed])


In [40]:
print(predicted_events)

['A man in a white shirt stands in front of a GMC logo and says "Replacing Your Tires."', 'A man is working on a car tire, using a tool to remove the tire', 'A hand holding a metal rod with a needle attached to it', 'A person is holding a coin and explaining something about it', 'A person is using a tool to remove a wheel from a truck', 'A person is standing next to a car with a tire on it', 'A man in a white shirt is shaking hands with another person in a black shirt', 'A man in a white shirt stands in front of a GMC logo and says "GMC Certified Service" and "Replacing Your Tires."']


In [42]:
#Get our annotations
ANNOTATIONS_PATH = "annotations/Video5_Nikitas_WithoutSound.txt"

video_annotations = []

#Read annotation file
with open(ANNOTATIONS_PATH, "r") as f:
    for line in f:
        #Split on comma and take the first element
        description = line.strip().split(",")[0]
        video_annotations.append(description)
        print(description)



Branding screen of GMC's tire replacement service
Person talking in front of a GMC car
Technician working on a car tire
Comparison of a tire with a good and bad thread
Displayed text recommending tire rotation every 7500 miles
Demonstration of checking tire tread depth using a tool and a coin
Tire spinning while mounted
People interacting near a car
Branding screen of GMC's tire replacement service






In [43]:

model_transformer = SentenceTransformer('all-MiniLM-L6-v2')

# embed ground truth and predicted event sentences
gt_embeddings = model_transformer.encode(video_annotations)
pred_embeddings = model_transformer.encode(predicted_events)


# computing pairwise cosine similarity between ground truth and predicted embeddings
sim_matrix = cosine_similarity(gt_embeddings, pred_embeddings) # maybe other way around




modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [44]:
print(sim_matrix)

[[ 0.67936677  0.25312114 -0.01895418  0.07492919  0.20909512  0.33703136
   0.05802464  0.701856  ]
 [ 0.48331457  0.20019636  0.06708275  0.30187112  0.28454018  0.4915387
   0.19325405  0.4762276 ]
 [ 0.3942943   0.66573024  0.05915316  0.1925776   0.43279254  0.53901577
   0.01244182  0.38993987]
 [ 0.30839086  0.37077156  0.04723415  0.10953654  0.22138727  0.3022503
   0.12251131  0.25720495]
 [ 0.35251588  0.24991208 -0.02688085  0.15837157  0.14771968  0.30974334
  -0.02420315  0.32528862]
 [ 0.28504705  0.41921276  0.04378822  0.42442936  0.31104594  0.3626367
  -0.01564999  0.26570418]
 [ 0.24904203  0.2866804   0.05624265  0.14911716  0.17361176  0.34963107
   0.04241404  0.1807718 ]
 [ 0.18739137  0.16411927  0.05140176  0.18711387  0.1729645   0.47729135
   0.24019426  0.17882463]
 [ 0.67936677  0.25312114 -0.01895418  0.07492919  0.20909512  0.33703136
   0.05802464  0.701856  ]
 [ 0.11109986  0.01319398  0.07280181  0.09871138 -0.00144555  0.00800359
   0.04278964  0.106

In [45]:
print(np.mean((sim_matrix)))

0.19284803


In [46]:
# THRESHOLD IS SOMETHING THAT CAN BE CHANGED (0.5 for now)

threshold = 0.5

matches = []
used_preds = set()


# THIS LOOP ENSURES THAT IF AN ANNOTATION IS MATCHED BY A PREDICTION IT CAN'T BE MATCHED BY ANOTHER ONE 
for i, gt in enumerate(video_annotations):
    
    # the predicted event that is most similar to a specific ground truth annotaiton
    best_j = np.argmax(sim_matrix[i])
    best_score = sim_matrix[i][best_j]

    # only a match if the predicted event is similar enough (higher than threshold) + it hasn't been matched to another ground truth 
    if best_score >= threshold and best_j not in used_preds: 
        matches.append((gt, predicted_events[best_j], best_score))
        used_preds.add(best_j)

In [47]:
# MAYBE NOT NEEDED

# NUMERICAL RESULTS (check if needed)

TP = len(matches)
FN = len(video_annotations) - TP
FP = len(predicted_events) - TP

precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * precision * recall / (precision + recall)\




In [48]:
print(precision)
print(recall)
print(f1)

0.25
0.15384615384615385
0.1904761904761905


In [49]:
# FINDING THE MISSED EVENTS (i.e if for a ground truth event no predicted was found to be similar)

matched_gt = set([m[0] for m in matches])

missed_events = [gt for gt in video_annotations if gt not in matched_gt]

print("\nMissed events:")
for e in missed_events:
    print("-", e)


Missed events:
- Person talking in front of a GMC car
- Comparison of a tire with a good and bad thread
- Displayed text recommending tire rotation every 7500 miles
- Demonstration of checking tire tread depth using a tool and a coin
- Tire spinning while mounted
- People interacting near a car
- 
- 
- 
- 


In [52]:
print(f"Model: {model_path}\n")

print(f"Video {video_number}")
print("Prompt given: ", PROMPT_EVENT_ONLY)

print(f"Chunk Length: {chunk_length}")
print(f"Max Tokens used (per Chunk) : {max_new_tokens}")
print(f"Number of Frames (per Chunk) :{num_frames}")

print(f"Threshold: {threshold}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}\n")

print("===\n")

for i in range(len(predicted_events)):
    print(f"Predicted event {i}:{predicted_events[i]}")






Model: HuggingFaceTB/SmolVLM2-256M-Video-Instruct

Video 5
Prompt given:   
Describe the most unique actions in this video segment in one concise sentence. Focus on what is happening and who might be involved. 

Chunk Length: 15
Max Tokens used (per Chunk) : 128
Number of Frames (per Chunk) :10
Threshold: 0.5
Precision: 0.250
Recall:    0.154
F1-score:  0.190

===

Predicted event 0:A man in a white shirt stands in front of a GMC logo and says "Replacing Your Tires."
Predicted event 1:A man is working on a car tire, using a tool to remove the tire
Predicted event 2:A hand holding a metal rod with a needle attached to it
Predicted event 3:A person is holding a coin and explaining something about it
Predicted event 4:A person is using a tool to remove a wheel from a truck
Predicted event 5:A person is standing next to a car with a tire on it
Predicted event 6:A man in a white shirt is shaking hands with another person in a black shirt
Predicted event 7:A man in a white shirt stands in fr

In [ ]:
text_file = []

text_file.append(f"Model: {model_path}\n")
text_file.append(f"Video {video_number}")
text_file.append(f"Prompt given: {PROMPT_EVENT_ONLY}")
text_file.append(f"Chunk Length: {chunk_length}")
text_file.append(f"Max Tokens used (per Chunk): {max_new_tokens}")
text_file.append(f"Number of Frames (per Chunk): {num_frames}")
text_file.append(f"Threshold: {threshold}")
text_file.append(f"Precision: {precision:.3f}")
text_file.append(f"Recall:    {recall:.3f}")
text_file.append(f"F1-score:  {f1:.3f}\n")
text_file.append("===\n")

for i, event in enumerate(predicted_events):
    text_file.append(f"Predicted event {i}: {event}")

os.makedirs("vlm_outputs", exist_ok=True)

#Change name
with open(f"vlm_outputs/{video_number}_name_prompt_indicator.txt", "a") as f:
    f.write("\n".join(text_file) + "\n")

['Model: HuggingFaceTB/SmolVLM2-256M-Video-Instruct\n', 'Video 5', 'Prompt given:  \nDescribe the most unique actions in this video segment in one concise sentence. Focus on what is happening and who might be involved. \n', 'Chunk Length: 15', 'Max Tokens used (per Chunk): 128', 'Number of Frames (per Chunk): 10', 'Threshold: 0.5', 'Precision: 0.250', 'Recall:    0.154', 'F1-score:  0.190\n', '===\n', 'Predicted event 0: A man in a white shirt stands in front of a GMC logo and says "Replacing Your Tires."', 'Predicted event 1: A man is working on a car tire, using a tool to remove the tire', 'Predicted event 2: A hand holding a metal rod with a needle attached to it', 'Predicted event 3: A person is holding a coin and explaining something about it', 'Predicted event 4: A person is using a tool to remove a wheel from a truck', 'Predicted event 5: A person is standing next to a car with a tire on it', 'Predicted event 6: A man in a white shirt is shaking hands with another person in a bl

## ADD STEP 4